# DZ14Questions — теоретичні запитання до логістичної регресії

У цьому ноутбуці відповіді на запитання з **Вправи 2** до задачі про манго та лимони. Використовуємо приклад з попереднього файлу `DZ14.ipynb`, де ми будували логістичну регресію за ознакою `sweet` для класифікації фруктів.

## Питання 1
**Чому логістична регресія краще підходить, ніж лінійна регресія для цієї вправи?**

### Відповідь
- У цій задачі цільова змінна — **клас** фрукта: `0` — лимон, `1` — манго. Тобто ми розв'язуємо **задачу класифікації**, а не прогнозування чисел.
- **Лінійна регресія** передбачає довільні дійсні числа (можуть бути менше 0, більше 1), і її результат не є ймовірністю. Якщо її застосувати, вона може повернути, наприклад, 1.3 або −0.4, що не має сенсу як ймовірність класу.
- **Логістична регресія** використовує сигмоїдну функцію, яка перетворює лінійну комбінацію ознаки на значення **від 0 до 1**. Це можна інтерпретувати як *ймовірність того, що фрукт — манго*.
- Далі ми просто застосовуємо поріг (наприклад, 0.5):
  - якщо ймовірність ≥ 0.5 → відносимо до класу `1` (манго),
  - якщо < 0.5 → до класу `0` (лимон).

**Висновок:** логістична регресія спеціально створена для задач, де вихід — це ймовірність належності до класу (0/1), тому вона природно підходить для задачі «манго чи лимон», а лінійна регресія тут була б неприродною і могла б давати некоректні значення.

## Питання 2
**Що відбувається, якщо всі зразки у вибірці мають однаковий клас (усі 1 або усі 0)?**

### Відповдь
- Якщо в датасеті **усі приклади одного класу** (наприклад, тільки манго, без жодного лимона), то модель **не бачить різниці** між класами. Вона не може навчитися, як ознака `sweet` впливає на поділ між 0 і 1, бо другого класу просто немає.
- У такій ситуації логістична регресія фактично вчиться **константній моделі**: 
  вона завжди буде повертати один і той самий клас (той, що є в датасеті), незалежно від значення `sweet`.
- У термінах ймовірностей це означає, що модель скаже щось на кшталт: 
  > ймовірність манго ≈ 1 (або 0), і ознака `sweet` ні на що не впливає.
- Якщо при цьому спробувати ділити дані на train/test, можливі попередження або помилки: у тренувальній чи тестовій вибірці може виявитися лише один клас, і деякі алгоритми класіфікації на це скаржаться.

**Висновок:** якщо в даних тільки один клас, модель не вчиться розрізняти класи, а просто завжди передбачає цей єдиний клас. Для нормального навчання потрібні приклади **обох** класів.

## Питання 3
**Чи може модель бути overfitted на такому маленькому наборі даних?**

### Відповідь
- У нас дуже **маленький датасет**: небагато прикладів манго та лимонів. Через це кожна окрема точка має **великий вплив** на результат навчання.
- Модель логістичної регресії сама по собі досить проста (лінійний роздільник + сигмоїда), тому вона не так сильно переобучається, як складні нейронні мережі.
- Але навіть така проста модель на дуже маленькому наборі може стати **нестійкою**: 
  - якщо ми додамо один «шумовий» приклад або приберемо одну точку, 
    коефіцієнти сильно зміняться, а разом з ними — ймовірності.
- Тобто формально **overfitting можливий** у тому сенсі, що модель ідеально запам'ятає невеликий набір і покаже 100% точність на train, але на нових даних може поводитися гірше.

**Висновок:** на такому маленькому датасеті модель може бути **дуже чутливою** до кожного спостереження. Це не класичний «жорсткий» overfitting, але ризик, що модель буде погано узагальнювати на нові дані, досить великий саме через маленький розмір вибірки.

## Питання 4
**Якщо модель вирішує "манго чи лимон" за солодкістю, чи можлива ситуація, коли модель дискримінує кислі манго?**

### Відповідь
- У нашій моделі є лише **одна ознака** — `sweet` (солодкість). Якщо манго зазвичай солодкі, а лимони — кислі, модель вивчить саме таку закономірність.
- Але припустимо, що існують **рідкісні кислі манго**: 
  тобто фрукти, у яких `sweet = 0`, але насправді `let = 1` (манго).
- Якщо таких прикладів мало або їх взагалі немає в навчальних даних, модель «вважатиме», що `sweet = 0` — це завжди лимон. Тоді реальні кислі манго вона буде класифікувати як лимони.
- У цьому сенсі можна сказати, що модель **"дискримінує" кислі манго**: вона не вміє їх правильно розпізнавати, бо вся логіка побудована лише на солодкості.

**Висновок:** так, така ситуація можлива. Якщо ми використовуємо тільки одну ознаку `sweet`, то всі незвичайні приклади (наприклад, кислий манго) будуть класифіковані неправильно. Моделі не вистачає інформації, щоб зрозуміти, що це все ще манго.

## Питання 5
**Якщо згенерувати 1 000 000 ідеальних даних для логістичної регресії, але 1% випадково переплутати — що буде з log-loss і вагами? Як сильно 1% шуму ламає навіть ідеальну логістичну модель?**

### Відповідь
- Уявімо, що в нас є 1 000 000 точок, які ідеально лежать на логістичній кривій. Якщо дані **без шуму**, модель може зробити практично **нульовий log-loss** і дуже круту сигмоїду, яка майже ідеально відділяє класи.
- Тепер 1% міток ми **переплутали** випадково. Це означає, що 10 000 точок мають неправильний клас (наприклад, лимон позначили як манго і навпаки).
- Логістична регресія намагається мінімізувати **log-loss**. 
  Якщо зробити ваги занадто великими (дуже крута межа), то ці 1% неправильних точок будуть давати **величезний вклад у помилку**, бо модель буде дуже впевнено помилятися на них.
- Тому оптимальні ваги для моделі з шумом будуть **меншими за модуль**, ніж у ідеальному випадку: сигмоїда стане трохи **менш крутою**, а log-loss вже не зможе бути нульовим — 
він буде невеликим, але **позитивним**.
- 1% шуму **не руйнує** логістичну модель, але:
  - заважає досягти ідеального log-loss;
  - змушує модель "компромісно" підлаштовуватися так, щоб помилки на шумових точках не були надто дорогими.

**Висновок:** 1% випадкового шуму не ламає модель кардинально, але робить неможливим ідеальне навчання (log-loss > 0), і трохи згладжує (пом'якшує) сигмоїду та значення ваг.

## Питання 6
**Як зміниться крива sigmoid, якщо додати одне "помилкове" спостереження? Наприклад, (0; 1) — шум у даних, який може бути зіпсованим манго?**

### Відповідь
- Розглянемо малий датасет, як у нашій задачі: 
  більшість фруктів з `sweet = 0` — це лимони (0), а з `sweet = 1` — манго (1).
- Якщо дані ідеальні (без шуму), логістична регресія спробує зробити **дуже круту сигмоїду**, яка майже точно відділяє `sweet = 0` від `sweet = 1`.
- Тепер додамо один "помилковий" приклад: `sweet = 0`, але `let = 1` — тобто **манго з нульовою солодкістю**.
- Для логістичної регресії цей один приклад стає **супер важливим**, тому що даних мало. Щоб не робити на ньому величезну помилку, модель змінює свої ваги так, щоб для `sweet = 0` ймовірність манго була не зовсім близькою до 0.
- На графіку це виглядає так:
  - сигмоїда стає **менш крутою**;
  - значення ймовірності для `sweet = 0` піднімається вище (наприклад, з ~0.1 до ~0.47, як ми бачили в задачі);
  - для `sweet = 1` ймовірність манго все одно залишається високою, але трохи зменшується.

**Висновок:** на маленькому наборі даних навіть **один шумовий приклад** (0; 1) може помітно змінити форму сигмоїди: вона стає м'якшою, а межа між класами — менш «жорсткою». Це показує, наскільки чутлива модель до шуму, коли даних мало.